# Restaurant Operations & Revenue Analytics - Data Cleaning Pipeline

## Business Problem
Small independent restaurants often manage orders, customers, payments, and deliveries across disconnected systems, resulting in incosistent and incomplete data. Without a clean, centralized dataset, it is impossible to reliably analyze revenue trends, customer behavior, or operational performance.

## Dataset
Synthetic dataset simulating 15 months of restaurant operations (Jan 2025 – Apr 2026), generated with realistic business patterns and intentional data quality issues.

**9 tables | ~7,000 orders | ~12,500 order line items**

| Table | Rows | Description |
|---|---|---|
| customers_raw | 126 | Customer contact info |
| employees_raw | 15 | Staff records |
| pizzas_raw | 25 | Menu items by size, crust, and category |
| toppings_raw | 15 | Available toppings with extra charges |
| pizza_toppings_raw | 82 | Pizza-topping combinations |
| orders_raw | ~6,900 | Transaction records |
| order_pizza_raw | ~12,500 | Line items linking orders to pizzas |
| payments_raw | ~6,000 | Payment records |
| deliveries_raw | ~5,200 | Delivery records |


In [102]:
import pandas as pd
import numpy as np

df_customers = pd.read_csv('data/raw/customers_raw.csv')
df_deliveries = pd.read_csv('data/raw/deliveries_raw.csv')
df_employees = pd.read_csv('data/raw/employees_raw.csv')
df_order_pizza = pd.read_csv('data/raw/order_pizza_raw.csv')
df_orders = pd.read_csv('data/raw/orders_raw.csv')
df_payments = pd.read_csv('data/raw/payments_raw.csv')
df_pizza_toppings = pd.read_csv('data/raw/pizza_toppings_raw.csv')
df_pizzas = pd.read_csv('data/raw/pizzas_raw.csv')
df_toppings = pd.read_csv('data/raw/toppings_raw.csv')



In [103]:
# ── CUSTOMERS DATA EXPLORATION 
print("SHAPE:", df_customers.shape)
print()
print(df_customers.head(10))
print()
print("DATA TYPES:")
print(df_customers.dtypes)
print()
print("MISSING VALUES:")
print(df_customers.isnull().sum())
print()
print("DUPLICATES:", df_customers.duplicated().sum())
print()
print("SAMPLE PHONE FORMATS:")
print(df_customers['Phone'].dropna().unique()[:15])
print()
print("NAME CASING SAMPLE:")
print(df_customers['Name'].head(20).tolist())

SHAPE: (126, 5)

   CustomerID                Name           Phone                     Email  \
0      770487          Sue Wilson      9143341057                       NaN   
1      191161      NANCY MORRISON    914 490 3402        mnunez@example.org   
2      850800        Aaron Wright      9145667265   vanessalong@example.com   
3      456778       Brent Miller       9142714803      fjohnson@example.com   
4      733052    KATELYN ANDERSON  (914) 309-4235         sford@example.net   
5      479201       William Henry             NaN     hwilliams@example.net   
6      996865        suzanne soto    914.712.0868  stephenglenn@example.com   
7      835911        Jacob Riddle    914 387 1230      joseph92@example.org   
8      497887       Travis Brooks    914-468-4531        lwhite@example.com   
9      340174  Jeffrey Mullen PhD    914.211.0460       julia14@example.org   

                                Address  
0           693 Barnes Dale, Pelham, NY  
1      575 Peter Harbors, Tar

In [104]:
# Customer Cleaning
df_customer_clean = df_customers.copy()

# White Spaces 
string_cols = ['Name', 'Email', 'Phone', 'Address']
df_customer_clean[string_cols] = df_customer_clean[string_cols].apply(lambda x: x.str.strip())

# Fix Name 
df_customer_clean['Name'] = df_customer_clean['Name'].str.title()

# Fix Phone numbers
df_customer_clean['Phone'] = df_customer_clean['Phone'].str.replace(r'\D','', regex=True)

# Nulls
df_customer_clean[ 'missing_email'] = df_customer_clean['Email'].isnull()
df_customer_clean['missing_phone'] = df_customer_clean['Phone'].isnull()

# Duplicate Customers
duplicate_customers = df_customer_clean.duplicated(subset=['Name', 'Address'], keep=False)

print("Duplicate Customers based on Name and Address:\n")
print(df_customer_clean[duplicate_customers][['CustomerID', 'Name', 'Address']])

print()
print("Cleaned Customers Data:\n")
print(df_customer_clean.head(10))


Duplicate Customers based on Name and Address:

     CustomerID             Name                            Address
20       847581    Alexis Miller  254 Bailey Plaza, Dobbs Ferry, NY
56       254511     David Forbes      425 Moyer Drive, Tuckahoe, NY
59       578634  Amanda Lawrence      737 Ford Club, Mamaroneck, NY
85       430876      Lisa Cannon    222 Joshua Place, Larchmont, NY
86       389930  Jessica Delgado    304 Sonya Rest, Dobbs Ferry, NY
115      993140  Michelle Hodges     37 Haney Trail, Bronxville, NY
120      109853      Lisa Cannon    222 Joshua Place, Larchmont, NY
121      943718  Michelle Hodges     37 Haney Trail, Bronxville, NY
122      401603  Amanda Lawrence      737 Ford Club, Mamaroneck, NY
123      673149  Jessica Delgado    304 Sonya Rest, Dobbs Ferry, NY
124      177449    Alexis Miller  254 Bailey Plaza, Dobbs Ferry, NY
125      462351     David Forbes      425 Moyer Drive, Tuckahoe, NY

Cleaned Customers Data:

   CustomerID                Name       Ph

In [105]:
#Drop duplicate customers
df_customer_clean = df_customer_clean.drop_duplicates(subset=['Name', 'Address'], keep='first')
print("Shape after dropping duplicates:", df_customer_clean.shape)

Shape after dropping duplicates: (120, 7)


## Customer Table - Cleaned
**Original shape:** 126 rows × 5 columns

**Issues identified and resolved:**

1. **Name casing** — inconsistent formatting found (ALL CAPS, all lowercase, mixed). 
   Resolved with `.str.title()` to standardize all names to Title Case.

2. **Whitespace** — leading and trailing spaces found in Name, Phone, Email, Address. 
   Resolved with `.str.strip()` across all string columns.

3. **Phone number formatting** — 5 different formats identified. 
Resolved by stripping all non-digit characters with regex — stored as raw 10-digit strings.

4. **Missing values** — 6 missing phone numbers, 20 missing emails. 
Not imputed — flagged with boolean columns `missing_phone` and `missing_email` to preserve data integrity.

5. **Near-duplicate customers** — 6 duplicate records identified sharing identical Name and Address with different CustomerIDs, likely from manual re-entry. 

**Clean shape:** 120 rows × 7 columns

In [106]:
# EMPLOYEES DATA EXPLORATION
print("Shape:", df_employees.shape)
print("\nHead:")
print(df_employees.head())
print("\nData Types:")
print(df_employees.dtypes)
print("\nMissing Values:")
print(df_employees.isnull().sum())
print("\nDuplicates:", df_employees.duplicated().sum())
print("\nRole values:")
print(df_employees['Role'].value_counts())
print("\nPhone sample:")
print(df_employees['Phone'].tolist())

Shape: (15, 4)

Head:
   EmployeeID              Name    Role           Phone
0      716113  Reginald Cochran    Cook  (914) 601-8975
1      823846         Ian White    Cook  (914) 519-5951
2      986365    Tiffany Melton    Cook    914.606.8760
3      503380    Nicole Bentley  Driver      9149106886
4      348728       Tanya Moore  Driver    914 739 8231

Data Types:
EmployeeID     int64
Name          object
Role          object
Phone         object
dtype: object

Missing Values:
EmployeeID    0
Name          0
Role          0
Phone         0
dtype: int64

Duplicates: 0

Role values:
Role
Driver        5
Cook          4
Cashier       2
Prep Cook     2
Manager       1
Shift Lead    1
Name: count, dtype: int64

Phone sample:
['(914) 601-8975', '(914) 519-5951', '914.606.8760', '9149106886', '914 739 8231', '914-596-3914', '914-764-0518', '914 182 2825', '914-933-3700', '914 486 0745', '914.691.0964', '914 418 9885', '9143294774', '914-166-1401', '9148395785']


In [107]:
# Employee Data Cleaning

df_employees_clean = df_employees.copy()

# Fix Phone numbers
df_employees_clean['Phone'] = df_employees_clean['Phone'].str.replace(r'\D', '', regex=True)

# Fix Name
df_employees_clean['Name'] = df_employees_clean['Name'].str.title()

print(df_employees_clean)

    EmployeeID                  Name        Role       Phone
0       716113      Reginald Cochran        Cook  9146018975
1       823846             Ian White        Cook  9145195951
2       986365        Tiffany Melton        Cook  9146068760
3       503380        Nicole Bentley      Driver  9149106886
4       348728           Tanya Moore      Driver  9147398231
5       476395         Tiffany Smith      Driver  9145963914
6       123005            Lisa Smith      Driver  9147640518
7       108491  Dr. Kimberly Johnson     Manager  9141822825
8       881369            Shari Hill     Cashier  9149333700
9       913423       Kelsey Marshall     Cashier  9144860745
10      940886           Paula Braun   Prep Cook  9146910964
11      767542        Vanessa Mendez   Prep Cook  9144189885
12      362801         Andrew Thomas  Shift Lead  9143294774
13      757937       Thomas Johnston      Driver  9141661401
14      926918   Mr. Jacob Rodriguez        Cook  9148395785


## Employee Table — Cleaning Summary

**Original shape:** 15 rows × 4 columns

**Issues identified and resolved:**

1. **Phone number formatting** — multiple formats found. Resolved by stripping all non-digit characters with regex.

**No issues found with:** nulls, duplicates, name casing, role consistency.

**Clean shape:** 15 rows × 4 columns

In [108]:
# DELIVERY DATA EXPLORATION
print("Shape:", df_deliveries.shape)
print("\nHead:")
print(df_deliveries.head(10))
print("\nMissing Values:")
print(df_deliveries.isnull().sum())
print("\nDuplicates:", df_deliveries.duplicated().sum())
print("\nStatus values:")
print(df_deliveries['Status'].value_counts())
print("\nAddressType values:")
print(df_deliveries['AddressType'].value_counts())

Shape: (5214, 5)

Head:
   DeliveryID  OrderID                                Address AddressType  \
0      475644   911984   809 Kennedy Squares, Dobbs Ferry, NY        Home   
1      546980   599380    703 Proctor Inlet, Mount Vernon, NY        Work   
2      550961   674993        463 Smith Isle, Dobbs Ferry, NY        Home   
3      688866   797416       576 Parsons Viaduct, Ardsley, NY       Other   
4      701987   401601         232 Jennifer Parkways, Rye, NY        Home   
5      385320   882354              454 Miller Alley, Rye, NY        Home   
6      649414   524707           81 Cruz Bridge, Harrison, NY        Work   
7      720148   896788        58 Samantha Ferry, Elmsford, NY        Home   
8      169113   141043   338 Michelle Drive, Mount Vernon, NY        Home   
9      802667   449555  720 Mccarthy Villages, Mamaroneck, NY        Home   

            Status  
0        DELIVERED  
1        Delivered  
2  Awaiting Pickup  
3        Delivered  
4        Delivered  
5 

In [109]:
# Delivery Data Cleaning
df_deliveries_clean = df_deliveries.copy()

# Fix Status 
df_deliveries_clean['Status'] = df_deliveries_clean['Status'].str.strip().str.title()

df_deliveries_clean['Status'] = df_deliveries_clean['Status'].str.replace('Out For Delivery', 'Out for Delivery')


# Confirm all values now
print(df_deliveries_clean['Status'].value_counts())

Status
Delivered           3630
Awaiting Pickup     1079
Out for Delivery     505
Name: count, dtype: int64


## Delivery Table — Cleaning Summary

**Issues identified and resolved:**

1. **Status casing inconsistencies** — 8 variations of 3 valid statuses found
   (e.g. "DELIVERED", "awaiting pickup", "Out For Delivery").
   Resolved with `.str.strip().str.title()` to standardize casing.


**Clean shape:** 5,214 rows × 5 columns

In [110]:
# ORDER_PIZZA DATA EXPLORATION 
print("Shape:", df_order_pizza.shape)
print("\nHead:")
print(df_order_pizza.head(10))
print("\nData Types:")
print(df_order_pizza.dtypes)
print("\nMissing Values:")
print(df_order_pizza.isnull().sum())
print("\nDuplicates:", df_order_pizza.duplicated().sum())
print("\nQuantity distribution:")
print(df_order_pizza['Quantity'].value_counts())
print("\nQuantity stats:")
print(df_order_pizza['Quantity'].describe())

Shape: (12481, 4)

Head:
   OrderPizzaID  OrderID  PizzaID  Quantity
0        102973   911984   194511         1
1        390743   911984   194511         2
2        722503   599380   409478         1
3        726410   674993   238579         1
4        656329   674993   883790         1
5        820184   797416   369452         1
6        597305   797416   409445         1
7        705991   401601   482584         2
8        380439   738577   867934         1
9        575682   882354   482584         1

Data Types:
OrderPizzaID    int64
OrderID         int64
PizzaID         int64
Quantity        int64
dtype: object

Missing Values:
OrderPizzaID    0
OrderID         0
PizzaID         0
Quantity        0
dtype: int64

Duplicates: 0

Quantity distribution:
Quantity
1    10529
2     1952
Name: count, dtype: int64

Quantity stats:
count    12481.000000
mean         1.156398
std          0.363247
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max    

In [111]:
# Check for same pizza appearing twice in same order
dupe_line_items = df_order_pizza[df_order_pizza.duplicated(subset=['OrderID', 'PizzaID'], keep=False)]
print(f"Duplicate OrderID + PizzaID combinations: {len(dupe_line_items)}")
print(dupe_line_items.head(10))

Duplicate OrderID + PizzaID combinations: 696
     OrderPizzaID  OrderID  PizzaID  Quantity
0          102973   911984   194511         1
1          390743   911984   194511         2
71         392559   739711   924718         1
72         685196   739711   924718         1
97         196955   446305   867934         1
98         661654   446305   867934         1
103        668471   277487   310548         1
104        772985   277487   310548         2
124        685098   203396   665610         1
126        821068   203396   665610         1


In [112]:
# Fixing quantity issues
df_order_pizza_clean = df_order_pizza.groupby(['OrderID', 'PizzaID'], as_index=False).agg(
    PizzaOrderID=('OrderPizzaID', 'first'),
    Quantity=('Quantity', 'sum')
)
print(f"Rows before: {len(df_order_pizza)}")
print(f"Rows after: {len(df_order_pizza_clean)}")
print(df_order_pizza_clean.head(10))

Rows before: 12481
Rows after: 12130
   OrderID  PizzaID  PizzaOrderID  Quantity
0   100037   284171        645073         2
1   100111   409445        133262         1
2   100209   442589        126420         1
3   100209   626169        425926         1
4   100573   238579        887335         1
5   100573   385765        428622         2
6   100573   665610        458441         1
7   100573   707727        418575         1
8   100725   656639        521906         1
9   100725   867934        161444         1


In [113]:
# ORDER DATA EXPLORATION
print("Shape:", df_orders.shape)
print("\nHead:")
print(df_orders.head(10))
print("\nData Types:")
print(df_orders.dtypes)
print("\nMissing Values:")
print(df_orders.isnull().sum())
print("\nDuplicates:", df_orders.duplicated().sum())
print("\nStatus values:")
print(df_orders['Status'].value_counts())
print("\nTotalPrice stats:")
print(df_orders['TotalPrice'].describe())
print("\nNegative prices:")
print(df_orders[df_orders['TotalPrice'] <= 0])

Shape: (6906, 7)

Head:
   OrderID  CustomerID  EmployeeID   OrderDate OrderTime  TotalPrice  \
0   911984      882169      503380  2026-02-04  11:52:55       50.97   
1   599380      682144      767542  2025-10-23  12:33:07       21.49   
2   674993      265080      913423  2025-06-13  17:08:17       46.23   
3   797416      269430      123005  2025-07-25  22:06:20       39.48   
4   401601      890870      767542  2025-03-24  20:55:58       29.48   
5   738577      926097      881369  2026-02-13  21:43:41       17.49   
6   882354      617488      716113  2025-09-10  18:47:40       31.73   
7   397005      916232      123005  2025-08-01  13:07:22       62.22   
8   284229      497519      767542  2026-01-10  11:00:31       53.72   
9   524707      395442      348728  2025-12-18  22:40:26       19.24   

      Status  
0  Delivered  
1  Delivered  
2   Received  
3  Delivered  
4  Delivered  
5  Delivered  
6  Delivered  
7  Delivered  
8  Delivered  
9   Deliverd  

Data Types:
Order

In [114]:
# Order Data Cleaning
df_orders_clean = df_orders.copy()

# Fix Date and Time
df_orders_clean['OrderDate'] = pd.to_datetime(df_orders_clean['OrderDate'])
df_orders_clean['OrderTime'] = pd.to_datetime(df_orders_clean['OrderTime'],format='%H:%M:%S').dt.time

# Fix Status
df_orders_clean['Status'] = df_orders_clean['Status'].str.strip().str.title()
status_corrctions = {
    'Deliverd': 'Delivered',
    'Canceld': 'Cancelled',
    'Recieved': 'Received',
    'Preparring': 'Preparing',
}
df_orders_clean['Status'] = df_orders_clean['Status'].replace(status_corrctions)

print(df_orders_clean['Status'].value_counts())
print("\nDate type:", df_orders_clean['OrderDate'].dtype)
print("Time type:", df_orders_clean['OrderTime'].dtype)


Status
Delivered    4817
Received      729
Cancelled     700
Preparing     660
Name: count, dtype: int64

Date type: datetime64[ns]
Time type: object


## Order Table — Cleaning Summary


**Issues identified and resolved:**

1. **Status casing inconsistencies** — 12 variations of 4 valid statuses found
   (e.g. "DELIVERED", "Deliverd", "Canceld", "Preparring", "CANCELLED").
   Resolved with `.str.strip().str.title()` followed by a manual mapping 
   dictionary to correct typos that survived title casing.

2. **Incorrect data types** — OrderDate and OrderTime stored as strings (object).
   OrderDate converted to `datetime64` with `pd.to_datetime()`.
   OrderTime converted to Python time objects with format `%H:%M:%S`.


In [115]:
# PAYMENTS DATA EXPLORATION
print("Shape:", df_payments.shape)
print("\nHead:")
print(df_payments.head(10))
print("\nData Types:")
print(df_payments.dtypes)
print("\nMissing Values:")
print(df_payments.isnull().sum())
print("\nDuplicates:", df_payments.duplicated().sum())
print("\nPayment Method values:")
print(df_payments['Method'].value_counts())
print("\nAmount stats:")
print(df_payments['Amount'].describe())


print("\nAmount mismatch with TotalPrice:")
merged = df_payments.merge(df_orders[['OrderID','TotalPrice']], on='OrderID', how='left')
merged['amount_diff'] = (merged['Amount'] - merged['TotalPrice']).round(2)
print(merged[merged['amount_diff'] != 0][['OrderID','Amount','TotalPrice','amount_diff']].head(10))
print(f"Total mismatches: {len(merged[merged['amount_diff'] != 0])}")

Shape: (6071, 6)

Head:
   PaymentID  OrderID PaymentDate PaymentTime  Amount       Method
0     215266   911984  2026-02-04    11:08:34   50.97         Cash
1     132663   599380  2025-10-23    12:39:02   21.49         Cash
2     782535   674993  2025-06-13    18:06:50   46.23  Credit Card
3     490748   797416  2025-07-25    22:05:27   39.48  Credit Card
4     467121   401601  2025-03-24    20:42:30   29.48   Debit Card
5     468074   738577  2026-02-13    22:44:48   17.49         CASH
6     263137   882354  2025-09-10    19:46:20   31.73  Credit Card
7     321494   397005  2025-08-01    13:25:38   61.43  Credit Card
8     582087   284229  2026-01-10    12:04:25   53.72  Credit Card
9     985506   524707  2025-12-18    23:53:44   19.24         Cash

Data Types:
PaymentID        int64
OrderID          int64
PaymentDate     object
PaymentTime     object
Amount         float64
Method          object
dtype: object

Missing Values:
PaymentID      0
OrderID        0
PaymentDate    0
Paymen

In [116]:
# Payment Data Cleaning
df_payments_clean = df_payments.copy()

# 1. Fix data types
df_payments_clean['PaymentDate'] = pd.to_datetime(df_payments_clean['PaymentDate'])
df_payments_clean['PaymentTime'] = pd.to_datetime(df_payments_clean['PaymentTime'], format='%H:%M:%S').dt.time

# 2. Fix Method name inconsistencies
df_payments_clean['Method'] = df_payments_clean['Method'].str.strip().str.title()

# 3. Flag amount mismatches
merged = df_payments_clean.merge(df_orders[['OrderID','TotalPrice']], on='OrderID', how='left')
df_payments_clean['amount_matches_order'] = (
    (df_payments_clean['Amount'] - merged['TotalPrice']).round(2) == 0
)


print(df_payments_clean['Method'].value_counts())
print(f"\nPayment amount mismatches flagged: {df_payments_clean['amount_matches_order'].eq(False).sum()}")

Method
Credit Card    2319
Cash           1480
Google Pay      771
Debit Card      768
Apple Pay       733
Name: count, dtype: int64

Payment amount mismatches flagged: 211


## Payment Table — Cleaning Summary

**Issues identified and resolved:**

1. **Method casing inconsistencies** — 10 variations of 5 valid payment methods found
   (e.g. "CASH", "CREDIT CARD", "GOOGLE PAY").
   Resolved with `.str.strip().str.title()`.

2. **Incorrect data types** — PaymentDate and PaymentTime stored as strings
   Converted to proper datetime and time types respectively.

3. **Amount mismatches** — 211 payment amounts do not match their corresponding 
   order TotalPrice. Discrepancies range from $0.12 to $1.94.
   Flagged with boolean column `amount_matches_order`


**Clean shape:** 6,071 rows × 7 columns

In [117]:
# PIZZA_TOPPINGS DATA EXPLORATION
print("Shape:", df_pizza_toppings.shape)
print("\nHead:")
print(df_pizza_toppings.head(10))
print("\nData Types:")
print(df_pizza_toppings.dtypes)
print("\nMissing Values:")
print(df_pizza_toppings.isnull().sum())
print("\nDuplicates:", df_pizza_toppings.duplicated().sum())
print("\nDuplicate PizzaID + ToppingID combinations:")
dupes = df_pizza_toppings[df_pizza_toppings.duplicated(subset=['PizzaID','ToppingID'], keep=False)]
print(f"Found: {len(dupes)}")

Shape: (82, 3)

Head:
   PizzaToppingID  PizzaID  ToppingID
0          684384   707727     455168
1          483542   707727     220764
2          194689   482584     455168
3          929090   482584     220764
4          513741   482584     943298
5          114594   867934     455168
6          377304   867934     220764
7          662636   867934     943298
8          229592   867934     591081
9          576877   237795     882824

Data Types:
PizzaToppingID    int64
PizzaID           int64
ToppingID         int64
dtype: object

Missing Values:
PizzaToppingID    0
PizzaID           0
ToppingID         0
dtype: int64

Duplicates: 0

Duplicate PizzaID + ToppingID combinations:
Found: 0


In [118]:
# PIZZAS DATA EXPLORATION
print("Shape:", df_pizzas.shape)
print("\nHead:")
print(df_pizzas.head(10))
print("\nData Types:")
print(df_pizzas.dtypes)
print("\nMissing Values:")
print(df_pizzas.isnull().sum())
print("\nDuplicates:", df_pizzas.duplicated().sum())
print("\nSize values:")
print(df_pizzas['Size'].value_counts())
print("\nCrustType values:")
print(df_pizzas['CrustType'].value_counts())
print("\nCategory values:")
print(df_pizzas['Category'].value_counts())
print("\nPrice stats:")
print(df_pizzas['Price'].describe())

Shape: (25, 6)

Head:
   PizzaID            Name    Size CrustType   Category  Price
0   707727      Margherita   Small      Thin    Classic  10.99
1   482584      Margherita  Medium      Thin    Classic  13.99
2   867934      Margherita   Large      Thin    Classic  15.99
3   237795       Pepperoni   Small     Thick    Classic  11.99
4   194511       Pepperoni  Medium     Thick    Classic  14.99
5   409445       Pepperoni   Large     Thick    Classic  16.99
6   442589     BBQ Chicken  Medium   Stuffed  Specialty  14.99
7   883790     BBQ Chicken   Large   Stuffed  Specialty  18.99
8   535686  Veggie Supreme   Small      Thin      Vegan  11.49
9   284171  Veggie Supreme   Large      Thin      Vegan  16.49

Data Types:
PizzaID        int64
Name          object
Size          object
CrustType     object
Category      object
Price        float64
dtype: object

Missing Values:
PizzaID      0
Name         0
Size         0
CrustType    0
Category     0
Price        0
dtype: int64

Duplicates:

In [119]:
# TOPPINGS DATA EXPLORATION
print("Shape:", df_toppings.shape)
print("\nHead:")
print(df_toppings)
print("\nData Types:")
print(df_toppings.dtypes)
print("\nMissing Values:")
print(df_toppings.isnull().sum())
print("\nDuplicates:", df_toppings.duplicated().sum())
print("\nExtraCharge values:")
print(df_toppings['ExtraCharge'].value_counts())

Shape: (15, 3)

Head:
    ToppingID         ToppingName  ExtraCharge
0      882824           Pepperoni         0.00
1      455168           Mushrooms         0.00
2      943298        Extra Cheese         1.50
3      220764        Black Olives         0.00
4      591081       Green Peppers         0.00
5      178957              Onions         0.00
6      247594           BBQ Sauce         0.50
7      890785           Jalapeños         0.75
8      336574               Bacon         1.50
9      809176           Pineapple         0.50
10     859396             Spinach         0.00
11     807166  Sun-dried Tomatoes         1.00
12     516716      Roasted Garlic         0.75
13     986827           Anchovies         1.00
14     943671         Red Peppers         0.00

Data Types:
ToppingID        int64
ToppingName     object
ExtraCharge    float64
dtype: object

Missing Values:
ToppingID      0
ToppingName    0
ExtraCharge    0
dtype: int64

Duplicates: 0

ExtraCharge values:
ExtraCharge
0

In [ ]:
# FINAL EXPORT — All 9 Clean Tables
import os
os.makedirs('data/clean', exist_ok=True)

# Fix topping special character before export
df_toppings['ToppingName'] = df_toppings['ToppingName'].str.replace('Jalapeños', 'Jalapenos')

# Drop flag columns — not needed in database
df_customer_clean = df_customer_clean.drop(columns=['missing_email', 'missing_phone'])
df_payments_clean = df_payments_clean.drop(columns=['amount_matches_order'])

# Fix order_pizza column name and order to match schema
df_order_pizza_clean = df_order_pizza_clean.rename(columns={'OrderPizzaID': 'PizzaOrderID'})
df_order_pizza_clean = df_order_pizza_clean[['PizzaOrderID', 'OrderID', 'PizzaID', 'Quantity']]

# Export all 9
df_customer_clean.to_csv('data/clean/customers_clean.csv', index=False)
df_employees_clean.to_csv('data/clean/employees_clean.csv', index=False)
df_deliveries_clean.to_csv('data/clean/deliveries_clean.csv', index=False)
df_order_pizza_clean.to_csv('data/clean/pizza_order_clean.csv', index=False)
df_orders_clean.to_csv('data/clean/orders_clean.csv', index=False)
df_payments_clean.to_csv('data/clean/payments_clean.csv', index=False)
df_pizza_toppings.to_csv('data/clean/pizza_toppings_clean.csv', index=False)
df_pizzas.to_csv('data/clean/pizzas_clean.csv', index=False)
df_toppings.to_csv('data/clean/toppings_clean.csv', index=False)

print("All 9 clean tables exported to data/clean/")

['PizzaOrderID', 'OrderID', 'PizzaID', 'Quantity']
   PizzaOrderID  OrderID  PizzaID  Quantity
0        645073   100037   284171         2
1        133262   100111   409445         1
2        126420   100209   442589         1
